# Getting Started
This notebook introduces the basic usage of **Granuscore**.

We start by importing the scorer.

In [1]:
from granuscore import GranuScore

granu_score = GranuScore()

In [5]:
from granuscore import GranuScore

granu_score2 = GranuScore(predictor_type='sentence_transformer',
                          faiss_index_path= '../assets/faiss/50k-all-min-index.faiss',
                          lgb_model_path='../assets/lgb_models/50k-all-min-nearest_neighbor_model.txt',
                          search_method='nearest_neighbor')

KeyboardInterrupt: 

## Scoring text
Below we score the texts from our visual abstract.
Lower Granuscore values indicate more fine-grained, specific references,
while higher values indicate coarser, more abstract language.

In [2]:
texts = [
    "Tony Hawk was born in San Diego",
    "Tony Hawk was born in California",
    "Tony Hawk was born in the United States",
    "A skateboarder was born in the United States",
    "A sportsman was born in the United States"
]

### Score a single text
`GranuScore` accepts either a single string or a list of strings.

In [7]:
print('Single Text\t\t', granu_score(texts[0]))

print('Multiple Texts\t', granu_score(texts))

Single Text		 29.322155
Multiple Texts	 [29.322155 40.344215 54.774826 74.53445  81.604416]


In [7]:
granu_score(['The skateboarder won a competition.', 'The skateboarder won a competition and set a new record.'] )

array([67.05983, 72.34051], dtype=float32)

### Inspect unit-level scores
Set `return_details=True` to return the split referential units and their individual scores, along with the mean-pooled overall score.

In [4]:
granu_score(texts[0], return_details=True)

{'text': 'Tony Hawk was born in San Diego',
 'pooled_score': np.float32(29.322155),
 'pooled_percentile': np.float32(29.322155),
 'bucket_output': 'low',
 'scopes': [{'scope_index': 0,
   'scope_text': 'Tony Hawk was born in San Diego',
   'pooled_score': np.float32(29.322155),
   'pooled_percentile': 29.322154454469594,
   'bucket_output': 'low',
   'unit_scores': {'Tony Hawk': np.float32(16.570896),
    'born': np.float32(48.917957),
    'San Diego': np.float32(22.477612)}}]}

### Example Sentences
We provide further examples illustrating how Granuscore works and how sensitive it is to changes in granularity levels.

In [5]:
example_texts = [
    'I bought a cappuccino at the small Italian café.',
    'I bought a warm milk drink at the small Italian café.',
    'I bought a drink at the small Italian café.',
    'I bought a drink at the café.',
    'I bought a drink at the restaurant.'
]

granu_score(example_texts, return_details=False)

array([47.20311 , 55.67821 , 61.812172, 69.93674 , 75.00651 ],
      dtype=float32)

In [6]:
example_texts = [
    "He fixed his CUBE road bike using a rusty wrench.",
    "He fixed his road bike using a rusty wrench.",
    "He fixed his bike using a rusty wrench.",
    "He fixed his bike using a wrench.",
    "He fixed his bike using a tool.",
]

granu_score(example_texts, return_details=False)

array([51.596184, 57.51634 , 67.23604 , 72.697716, 78.14763 ],
      dtype=float32)

In [7]:
example_texts = [
    'He sits on his old wooden chair.',
    'He sits on his wooden chair.',
    'He sits on a chair.',
    'He sits on furniture.'
]

granu_score(example_texts, return_details=False)

array([45.55225 , 56.667564, 69.29589 , 75.16452 ], dtype=float32)

### Batching
Granuscore supports batching at two different stages of the scoring pipeline.
The ``batch_size`` parameter controls how many input texts are processed
simultaneously. The ``encoding_batch_size`` parameter controls how many
referential units are encoded and searched in the FAISS index at the same time.

By default, ``encoding_batch_size`` is set to ``None``, meaning that all
referential units within a batch are processed together, with no explicit
limit. Setting this value to a fixed number (e.g., 128) can reduce memory usage
and may improve runtime performance for large batches or long texts.

In [11]:
granu_score(texts * 200, batch_size=1, show_progress_bar=True).mean()

Granuscore: 100%|██████████| 1000/1000 [00:04<00:00, 234.82it/s]


np.float32(56.116016)

In [12]:
granu_score(texts * 200, batch_size=32, show_progress_bar=True).mean()

Granuscore: 100%|██████████| 32/32 [00:04<00:00,  7.72it/s]


np.float32(56.116016)

In [8]:
granu_score(texts * 200, batch_size=32, encoding_batch_size=128, show_progress_bar=True).mean()

Granuscore: 100%|██████████| 32/32 [00:04<00:00,  7.95it/s]


np.float32(56.116016)

### Pooling

Granuscore supports several pooling strategies for aggregating referential unit
scores into a single text-level score. Below we illustrate some of these
strategies in practice.

The *pooling scope* determines the structural level at which aggregation is
performed. When the pooling scope is set to `sentence`, referential unit scores
are first aggregated within each sentence. The final text-level score is then
obtained by aggregating the resulting sentence-level scores.

By default, Granuscore uses:
- `pooling_scope='sentence'`
- `scope_pooling='lower_quantile_mean'` with `scope_tail_q=0.8`
- `pooling='mean'`

In [9]:
multiple_sentences_fine = "Tony Hawk was born in San Diego. He skates professionally in tournaments."
multiple_sentences_coarse = "A skateboarder was born in America. He skates a lot in tournaments."
multiple_sentences = [multiple_sentences_fine, multiple_sentences_coarse]

In [11]:
granu_score.predict(multiple_sentences, pooling_scope='document', pooling='mean',  return_details=False)

array([61.36986 , 70.677284], dtype=float32)

In [12]:
granu_score(multiple_sentences, pooling_scope='sentence', pooling='mean', return_details=False)

array([51.93299, 71.90253], dtype=float32)

In [13]:
granu_score(multiple_sentences, pooling_scope='document', pooling='lower_quantile_mean', tail_q=0.4, return_details=False)

array([31.123041, 58.41188 ], dtype=float32)

In [14]:
granu_score(multiple_sentences, pooling_scope='sentence', pooling='lower_quantile_mean', tail_q=0.4, return_details=False)

array([41.111362, 61.35789 ], dtype=float32)

### Score without splitting
If you already provide atomic units (e.g., entity strings), you can disable splitting with `split=False`.

In [2]:
granu_score(['Berlin', 'Germany'], split=False)

array([ 9.062957, 99.119576], dtype=float32)

### Datasets

In [20]:
from datasets import load_dataset

ds = load_dataset("rotten_tomatoes", split="train").take(10)

In [21]:
ds = ds.map(
    lambda x: {
        "granuscore": granu_score.predict(
            x["text"],
            batched=False,
            encoding_batch_size=128,
            show_progress_bar=False,
        )
    },
    batched=True,
    batch_size=4
)
ds

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'granuscore'],
    num_rows: 10
})